# 一、 多层感知机的简洁实现

## 1.库的引入

In [13]:
import torch
from torch import nn
from d2l import torch as d2l

## 2. 模型 + 初始化

In [14]:
net = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std = 0.01)
net.apply(init_weights)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=10, bias=True)
)

## 3.损失+训练

In [15]:
batch_size, lr, num_epochs = 256, 0.1, 10
# 4.损失 
loss = nn.CrossEntropyLoss(reduction = 'none')
trainer = torch.optim.SGD(net.parameters(), lr = lr)
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)


# 5. 训练循环（注意：l.mean().backward()，不是 sum！）
def train_epoch(net, train_iter, loss, updater):
    total_loss, total_acc, n = 0.0, 0.0, 0
    for X, y in train_iter:
        y_hat = net(X)
        l = loss(y_hat, y)
        updater.zero_grad()
        l.mean().backward()      # ← 关键：mean，不是 sum
        updater.step()
        total_loss += l.sum().item()   # 统计时用 sum 没问题
        total_acc += (y_hat.argmax(1) == y).sum().item()
        n += y.shape[0]
    return total_loss / n, total_acc / n
def evaluate_accuracy(net, data_iter):
    acc, n = 0.0, 0
    for X, y in data_iter:
        acc += (net(X).argmax(1) == y).sum().item()
        n += y.shape[0]
    return acc / n
def train(net, train_iter, test_iter, loss, num_epochs, updater):
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
        print(f'epoch {epoch + 1}: loss={train_loss:.4f}, train_acc={train_acc:.4f}, test_acc={test_acc:.4f}')

train(net, train_iter, test_iter,loss, num_epochs, trainer)

epoch 1: loss=1.0430, train_acc=0.6402, test_acc=0.7522
epoch 2: loss=0.5969, train_acc=0.7896, test_acc=0.8018
epoch 3: loss=0.5184, train_acc=0.8188, test_acc=0.8021
epoch 4: loss=0.4832, train_acc=0.8301, test_acc=0.8079
epoch 5: loss=0.4574, train_acc=0.8391, test_acc=0.8183
epoch 6: loss=0.4338, train_acc=0.8477, test_acc=0.8304
epoch 7: loss=0.4188, train_acc=0.8529, test_acc=0.8385
epoch 8: loss=0.4043, train_acc=0.8571, test_acc=0.8371
epoch 9: loss=0.3915, train_acc=0.8625, test_acc=0.8424
epoch 10: loss=0.3821, train_acc=0.8650, test_acc=0.8543


## 4.动手实验 A：验证 Torch-8 的结论
删掉 ReLU，两层 Linear 叠两层 Linear = 数学上等价于一层。
预测：训练会崩或者准确率大幅下降。

In [23]:
net_no_relu = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.Linear(256, 10))
net_no_relu.apply(init_weights)
loss = nn.CrossEntropyLoss(reduction='none')
trainer = torch.optim.SGD(net_no_relu.parameters(), lr=lr)
train(net_no_relu, train_iter, test_iter, loss, num_epochs, trainer)

epoch 1: loss=0.9628, train_acc=0.6808, test_acc=0.7712
epoch 2: loss=0.5683, train_acc=0.8033, test_acc=0.7994
epoch 3: loss=0.5119, train_acc=0.8213, test_acc=0.8125
epoch 4: loss=0.4835, train_acc=0.8310, test_acc=0.8115
epoch 5: loss=0.4682, train_acc=0.8366, test_acc=0.7666
epoch 6: loss=0.4572, train_acc=0.8398, test_acc=0.8322
epoch 7: loss=0.4447, train_acc=0.8452, test_acc=0.8340
epoch 8: loss=0.4403, train_acc=0.8455, test_acc=0.8321
epoch 9: loss=0.4352, train_acc=0.8479, test_acc=0.8201
epoch 10: loss=0.4284, train_acc=0.8507, test_acc=0.8390


## 5.调参实验
原参数：1 个隐藏层（256 单元），lr=0.1。
试 2 个配置，记录准确率：
- 配置 B：隐藏层 256 → lr=0.5
- 配置 C：两个隐藏层 [128, 64]
思考：调 lr 和加深网络，哪个对结果影响大？

In [24]:
# 配置 B
net_b = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
net_b.apply(init_weights)
loss = nn.CrossEntropyLoss(reduction = 'none')
trainer = torch.optim.SGD(net_b.parameters(), lr = 0.5)
train(net_b, train_iter, test_iter, loss, num_epochs, trainer)

epoch 1: loss=0.8025, train_acc=0.7010, test_acc=0.7789
epoch 2: loss=0.4817, train_acc=0.8223, test_acc=0.7452
epoch 3: loss=0.4238, train_acc=0.8430, test_acc=0.8431
epoch 4: loss=0.3908, train_acc=0.8558, test_acc=0.8247
epoch 5: loss=0.3638, train_acc=0.8654, test_acc=0.8322
epoch 6: loss=0.3486, train_acc=0.8712, test_acc=0.8307
epoch 7: loss=0.3368, train_acc=0.8748, test_acc=0.8399
epoch 8: loss=0.3238, train_acc=0.8803, test_acc=0.8461
epoch 9: loss=0.3161, train_acc=0.8828, test_acc=0.8699
epoch 10: loss=0.3016, train_acc=0.8888, test_acc=0.8484


In [27]:
# 配置 C
net_c = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                      nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
net_c.apply(init_weights)
loss = nn.CrossEntropyLoss(reduction='none')
trainer = torch.optim.SGD(net_c.parameters(), lr=lr)
train(net_c, train_iter, test_iter, loss, num_epochs, trainer)

epoch 1: loss=2.1628, train_acc=0.1610, test_acc=0.4278
epoch 2: loss=1.0664, train_acc=0.5853, test_acc=0.6656
epoch 3: loss=0.8065, train_acc=0.6951, test_acc=0.7282
epoch 4: loss=0.6853, train_acc=0.7499, test_acc=0.7696
epoch 5: loss=0.5890, train_acc=0.7860, test_acc=0.7510
epoch 6: loss=0.5351, train_acc=0.8088, test_acc=0.8062
epoch 7: loss=0.4947, train_acc=0.8249, test_acc=0.8203
epoch 8: loss=0.4634, train_acc=0.8347, test_acc=0.8307
epoch 9: loss=0.4433, train_acc=0.8408, test_acc=0.8354
epoch 10: loss=0.4227, train_acc=0.8472, test_acc=0.8322


## 6.实验 C2

In [29]:
# 换成 He 初始化（专为 ReLU 设计），看第一轮是否恢复正常
def init_he(m):
    if type(m) == nn.Linear:
        nn.init.kaiming_normal_(m.weight, nonlinearity = 'relu')
net_c2 = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
net_c2.apply(init_he)
loss = nn.CrossEntropyLoss(reduction = 'none')
trainer = torch.optim.SGD(net_c2.parameters(), lr = 0.1)
train(net_c2, train_iter, test_iter, loss, num_epochs, trainer)

epoch 1: loss=0.7410, train_acc=0.7415, test_acc=0.7743
epoch 2: loss=0.5069, train_acc=0.8202, test_acc=0.8053
epoch 3: loss=0.4505, train_acc=0.8384, test_acc=0.8369
epoch 4: loss=0.4166, train_acc=0.8494, test_acc=0.8297
epoch 5: loss=0.3970, train_acc=0.8570, test_acc=0.8530
epoch 6: loss=0.3766, train_acc=0.8646, test_acc=0.8468
epoch 7: loss=0.3648, train_acc=0.8673, test_acc=0.8359
epoch 8: loss=0.3526, train_acc=0.8729, test_acc=0.8510
epoch 9: loss=0.3429, train_acc=0.8749, test_acc=0.8528
epoch 10: loss=0.3343, train_acc=0.8780, test_acc=0.8448


## 6.小结

五组对比（10 轮）：

初始（256, ReLU, lr=0.1）：test 85.4%，泛化最好。<br>
实验 B（lr=0.5）：train 88.9% / test 84.8%，收敛快但 gap 4.1%，过拟合。<br>
实验 C（128→64, std=0.01）：test 83.2%，且第一轮 train_acc 仅 16%,loss≈ln(10)——std=0.01 初始化在 3 层网络上信号衰减，无法学习。<br>
C2（128→64, He 初始化）：第一轮恢复 74%，但 test 84.5% 仍低于初始，gap 3.3%——任务容量已饱和，加深无收益且更易过拟合。<br>
实验 A（无 ReLU）：test 83.9%，仍是线性模型，与 3.7 softmax 持平。 

(1)评判训练结果好坏看 test_acc；<br>
(2)lr 过大伤泛化；<br>
(3)加深网络在简单任务上无效，深度增加必须换匹配的初始化,且 std=0.01 初始化不适用于更深的网络，需换 He 初始化（即c2）。

拟合 (fit)<br>
模型在训练集上学懂数据的能力<br>
拟合好：在训练数据上预测很准，能抓住训练数据里的规律。<br>
过拟合 overfit：死记训练集，把噪声、随机波动也当成规律；训练集效果极好，一到新数据就崩。<br>
欠拟合 underfit：模型太弱，训练集本身就学不好，两边效果都差。

泛化 (generalization)<br>
模型在从未见过的新测试数据上表现良好的能力。<br>
泛化能力 = 模型能不能把训练学到的规律，用到陌生样本上。

拟合：记住训练数据<br>
泛化：看懂陌生新数据